# LeetCode 276 — Paint Fence bằng Python

**Link đề:** https://leetcode.com/problems/paint-fence/
**Độ khó:** Easy (theo danh mục repo) · **Chủ đề:** Dynamic Programming

> Ghi chú độ khó: LeetCode hiện ghi bài này là **Medium**; repo xếp vào **Easy** nên notebook lưu trong `DSA/Easy`.

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def numWays(self, n: int, k: int) -> int:
        ...
```

## Mục tiêu học
1. Đếm tổ hợp có **ràng buộc** bằng cách liệt kê (brute force) để hiểu đề thật chắc.
2. Nhận ra cấu trúc con lặp lại → **quy hoạch động (DP)**: định nghĩa trạng thái, công thức chuyển, giá trị đầu.
3. Hai cách mô hình: trạng thái `same/diff` và công thức gộp `total[i] = (k-1)·(total[i-1] + total[i-2])`.
4. Viết DP theo 3 kiểu: đệ quy + memo (`lru_cache`), bảng (tabulation), và **tối ưu bộ nhớ** `O(1)`.
5. Trace bảng DP, chứng minh công thức, và mở rộng sang các biến thể (tối đa `m` cột liên tiếp cùng màu).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Có một hàng rào gồm `n` cột (post), và `k` màu sơn. Sơn mỗi cột **đúng một** màu sao cho **không có 3 cột liên tiếp trở lên cùng màu**. Trả về số cách sơn.

**Ví dụ 1:** `n = 3, k = 2` → `6`. Với 2 màu `R`, `G` có `2^3 = 8` cách, trừ `RRR` và `GGG` → còn 6.

**Ví dụ 2:** `n = 1, k = 1` → `1`.

**Ví dụ 3:** `n = 7, k = 2` → `42`.

**Constraints (theo hiểu biết về đề):** `1 <= n <= 50`, `1 <= k <= 10^5`; đáp án đảm bảo nằm trong `[0, 2^31 - 1]`.

Chú ý: **2 cột liên tiếp cùng màu thì được**, chỉ cấm 3.

In [1]:
all_colorings = [a + b_ + c for a in "RG" for b_ in "RG" for c in "RG"]
valid = [s for s in all_colorings if not (s[0] == s[1] == s[2])]
print("Tất cả:", all_colorings)
print("Hợp lệ:", valid, "-> số cách =", len(valid))

Tất cả: ['RRR', 'RRG', 'RGR', 'RGG', 'GRR', 'GRG', 'GGR', 'GGG']
Hợp lệ: ['RRG', 'RGR', 'RGG', 'GRR', 'GRG', 'GGR'] -> số cách = 6


## 2. Nền tảng Python

### 2.1. `itertools.product` — liệt kê mọi tổ hợp

`product(range(k), repeat=n)` sinh mọi tuple độ dài `n`, mỗi phần tử là một màu `0..k-1`. Có `k^n` tuple.

In [2]:
from itertools import product
print(list(product(range(2), repeat=3)))
print("số tuple:", len(list(product(range(3), repeat=4))), "= 3^4 =", 3 ** 4)

[(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)]
số tuple: 81 = 3^4 = 81


### 2.2. Kiểm tra "3 phần tử liên tiếp bằng nhau"

Duyệt `i` từ `2` tới `n-1`, so `c[i] == c[i-1] == c[i-2]`. Python cho phép **so sánh dây chuyền** `a == b == c` (nghĩa là `a == b and b == c`). `any(...)` trả `True` nếu có ít nhất một phần tử đúng.

In [3]:
def has_three_in_row(c) -> bool:
    return any(c[i] == c[i - 1] == c[i - 2] for i in range(2, len(c)))

print(has_three_in_row((0, 0, 1, 1)), has_three_in_row((0, 1, 1, 1)), has_three_in_row((5,)))

False True False


### 2.3. Đệ quy và ghi nhớ với `functools.lru_cache`

`@lru_cache(maxsize=None)` đặt trên một hàm sẽ **lưu kết quả** theo tham số: lần sau gọi lại với cùng tham số thì trả ngay, không tính lại. Đây là kỹ thuật **memoization**. Ví dụ Fibonacci: không memo thì gọi hàm theo cấp số mũ, có memo thì tuyến tính.

In [4]:
from functools import lru_cache
calls = 0
def fib_plain(i):
    global calls
    calls += 1
    return i if i < 2 else fib_plain(i - 1) + fib_plain(i - 2)
fib_plain(25)
print("Không memo, số lần gọi:", calls)

calls = 0
@lru_cache(maxsize=None)
def fib_memo(i):
    global calls
    calls += 1
    return i if i < 2 else fib_memo(i - 1) + fib_memo(i - 2)
fib_memo(25)
print("Có memo, số lần gọi:", calls)

Không memo, số lần gọi: 242785
Có memo, số lần gọi: 26


### 2.4. DP là gì (nói ngắn)

**Dynamic Programming** = chia bài toán lớn thành bài toán con **cùng dạng, nhỏ hơn**, tính mỗi bài con **một lần**, rồi ghép lại. Ba câu hỏi luôn phải trả lời:
1. **Trạng thái**: `dp[i]` nghĩa là gì?
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp` nào trước đó?
3. **Giá trị đầu** (base case) và **đáp án** nằm ở đâu?

## 3. Cách 1 — Brute force: liệt kê `k^n` cách

Duyệt mọi cách sơn, đếm cách không có 3 cột liên tiếp cùng màu. Đúng tuyệt đối → dùng làm **oracle** để kiểm tra các cách sau. Nhưng `k^n` bùng nổ: `n = 50, k = 2` đã là `2^50 ≈ 10^15`.

In [5]:
def num_ways_brute(n: int, k: int) -> int:
    return sum(1 for c in product(range(k), repeat=n) if not has_three_in_row(c))

print(num_ways_brute(3, 2), num_ways_brute(1, 1), num_ways_brute(7, 2))
print("2^50 =", f"{2 ** 50:,}", "cách cần liệt kê nếu n = 50, k = 2")

6 1 42
2^50 = 1,125,899,906,842,624 cách cần liệt kê nếu n = 50, k = 2


## 4. Nút thắt và insight — nhìn vào **cột cuối**

Brute force tính lại cùng một "đoạn đầu" vô số lần. Hãy hỏi: *khi thêm cột `i`, điều gì quyết định nó có hợp lệ không?* Chỉ **hai cột trước** quan trọng.

Chia các cách sơn hợp lệ `i` cột thành hai loại theo **hai cột cuối**:
- `same[i]`: cột `i` **cùng màu** cột `i-1`.
- `diff[i]`: cột `i` **khác màu** cột `i-1`.

Chuyển trạng thái:
- `same[i] = diff[i-1]`: muốn cột `i` cùng màu cột `i-1`, thì cột `i-1` **phải khác** cột `i-2` (nếu không sẽ thành 3 cột cùng màu). Và chỉ có **1** cách chọn màu (copy màu cột `i-1`).
- `diff[i] = (same[i-1] + diff[i-1]) · (k - 1)`: từ **bất kỳ** cách hợp lệ `i-1` cột, chọn màu khác cột `i-1` → `k-1` lựa chọn.

Giá trị đầu: `same[1] = 0`, `diff[1] = k` (cột đầu tiên coi như "khác" cột không tồn tại).

Đáp án: `same[n] + diff[n]`.

In [6]:
k = 2
same, diff = 0, k
print(f"i=1: same = {same}, diff = {diff}, total = {same + diff}")
for i in range(2, 5):
    same, diff = diff, (same + diff) * (k - 1)
    print(f"i={i}: same = {same}, diff = {diff}, total = {same + diff}")

i=1: same = 0, diff = 2, total = 2
i=2: same = 2, diff = 2, total = 4
i=3: same = 2, diff = 4, total = 6
i=4: same = 4, diff = 6, total = 10


### 4.1. Gộp thành một công thức

Đặt `total[i] = same[i] + diff[i]`. Thay vào:

```text
total[i] = same[i] + diff[i]
         = diff[i-1] + (k-1)·total[i-1]
         = (k-1)·total[i-2] + (k-1)·total[i-1]      (vì diff[i-1] = (k-1)·total[i-2])
         = (k-1) · (total[i-1] + total[i-2])
```

Giá trị đầu: `total[1] = k`, `total[2] = k·k` (2 cột tùy ý, chưa thể có 3 cột trùng).

Diễn giải trực tiếp: cột `i` **khác** cột `i-1` → `(k-1)·total[i-1]`; cột `i` **cùng** cột `i-1` → khi đó cột `i-1` phải khác cột `i-2` → `(k-1)·total[i-2]`.

In [7]:
k = 2
t = {1: k, 2: k * k}
for i in range(3, 8):
    t[i] = (k - 1) * (t[i - 1] + t[i - 2])
print(t)

{1: 2, 2: 4, 3: 6, 4: 10, 5: 16, 6: 26, 7: 42}


## 5. Cách 2 — Đệ quy + memo (top-down)

Viết đúng công thức như một hàm đệ quy, thêm `lru_cache` để mỗi `total(i)` chỉ tính một lần.

In [8]:
def num_ways_memo(n: int, k: int) -> int:
    @lru_cache(maxsize=None)
    def total(i: int) -> int:
        if i == 1:
            return k
        if i == 2:
            return k * k
        return (k - 1) * (total(i - 1) + total(i - 2))
    return total(n)

print(num_ways_memo(3, 2), num_ways_memo(1, 1), num_ways_memo(7, 2), num_ways_memo(30, 2))

6 1 42 2692538


## 6. Cách 3 — Lời giải chính: DP bảng (bottom-up), giảng từng đoạn

**Đoạn 1 — biên:** `if n == 1: return k`. (Cũng có thể xử lý `n == 2` riêng, nhưng bảng tự lo được.)

**Đoạn 2 — tạo bảng:** `dp = [0] * (n + 1)` — list dài `n+1` toàn 0, dùng chỉ số `1..n` cho dễ đọc. `dp[1] = k`, `dp[2] = k * k`.

**Đoạn 3 — điền bảng:** `for i in range(3, n + 1): dp[i] = (k - 1) * (dp[i - 1] + dp[i - 2])`.

**Đoạn 4:** `return dp[n]`.

In [9]:
def num_ways_table(n: int, k: int) -> int:
    if n == 1:
        return k
    dp = [0] * (n + 1)
    dp[1] = k
    dp[2] = k * k
    for i in range(3, n + 1):
        dp[i] = (k - 1) * (dp[i - 1] + dp[i - 2])
    return dp[n]

print(num_ways_table(3, 2), num_ways_table(1, 1), num_ways_table(7, 2))

6 1 42


## 7. Cách 4 — Tối ưu bộ nhớ: chỉ giữ 2 giá trị gần nhất

`dp[i]` chỉ cần `dp[i-1]` và `dp[i-2]` → giữ hai biến `prev2`, `prev1` rồi "trượt" dần. Phép gán đồng thời `a, b = b, c` của Python tính **vế phải trước** rồi mới gán, nên không cần biến tạm.

In [10]:
def num_ways_o1(n: int, k: int) -> int:
    if n == 1:
        return k
    prev2, prev1 = k, k * k          # total[1], total[2]
    for _ in range(3, n + 1):
        prev2, prev1 = prev1, (k - 1) * (prev1 + prev2)
    return prev1

print(num_ways_o1(3, 2), num_ways_o1(1, 1), num_ways_o1(7, 2))
a, b_ = 1, 2
a, b_ = b_, a + b_
print("sau a, b = b, a + b:", a, b_)

6 1 42
sau a, b = b, a + b: 2 3


## 8. TRACE bảng DP

Với `n = 5, k = 3`:

| i | same[i] | diff[i] | total[i] | kiểm tra công thức gộp |
|---:|---:|---:|---:|---|
| 1 | 0 | 3 | 3 | — |
| 2 | 3 | 6 | 9 | `3·3 = 9` |
| 3 | 6 | 18 | 24 | `2·(9 + 3) = 24` |
| 4 | 18 | 48 | 66 | `2·(24 + 9) = 66` |
| 5 | 48 | 132 | 180 | `2·(66 + 24) = 180` |

Ý nghĩa cột `same[3] = 6`: số cách 3 cột mà cột 3 cùng màu cột 2 = số cách 2 cột mà cột 2 khác cột 1 = `diff[2] = 6`.

In [11]:
def trace_paint_fence(n: int, k: int) -> int:
    same, diff = 0, k
    print(f" i | same | diff | total")
    print(f" 1 | {same:>4} | {diff:>4} | {same + diff:>5}")
    for i in range(2, n + 1):
        new_same = diff
        new_diff = (same + diff) * (k - 1)
        same, diff = new_same, new_diff
        print(f"{i:>2} | {same:>4} | {diff:>4} | {same + diff:>5}   (same = diff cũ, diff = total cũ × {k - 1})")
    return same + diff

ans = trace_paint_fence(5, 3)
print("Đáp án:", ans, "| brute force:", num_ways_brute(5, 3))

 i | same | diff | total
 1 |    0 |    3 |     3
 2 |    3 |    6 |     9   (same = diff cũ, diff = total cũ × 2)
 3 |    6 |   18 |    24   (same = diff cũ, diff = total cũ × 2)
 4 |   18 |   48 |    66   (same = diff cũ, diff = total cũ × 2)
 5 |   48 |  132 |   180   (same = diff cũ, diff = total cũ × 2)
Đáp án: 180 | brute force: 180


### 8.1. Nhìn tận mắt các cách sơn được đếm (n = 3, k = 2)

Phân loại 6 cách hợp lệ theo `same/diff` ở cột cuối:

In [12]:
for c in product("RG", repeat=3):
    if has_three_in_row(c):
        continue
    kind = "same" if c[-1] == c[-2] else "diff"
    print("".join(c), "->", kind)
print("same[3] =", 2, "= diff[2] =", 2, "| diff[3] =", 4, "= total[2] * (k-1) =", 4 * 1)

RRG -> diff
RGR -> diff
RGG -> same
GRR -> same
GRG -> diff
GGR -> diff
same[3] = 2 = diff[2] = 2 | diff[3] = 4 = total[2] * (k-1) = 4


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def numWays(self, n: int, k: int) -> int:
        if n == 1:
            return k
        prev2, prev1 = k, k * k
        for _ in range(3, n + 1):
            prev2, prev1 = prev1, (k - 1) * (prev1 + prev2)
        return prev1

sol = Solution()
print(sol.numWays(3, 2), sol.numWays(1, 1), sol.numWays(7, 2))

6 1 42


**Lưu ý phiên bản cũ của đề:** từng có ràng buộc `n >= 0` và test `n = 0` (đáp án `0`). Nếu muốn an toàn, thêm `if n == 0: return 0` ở đầu — không ảnh hưởng các test khác.

## 10. Vì sao đúng? (chứng minh ngắn) + độ phức tạp

Mọi cách sơn hợp lệ `i` cột (`i >= 3`) rơi vào **đúng một** trong hai nhóm:
1. **Cột `i` khác màu cột `i-1`.** Bỏ cột `i` đi, phần còn lại là một cách hợp lệ `i-1` cột bất kỳ. Ngược lại, từ mỗi cách hợp lệ `i-1` cột, có đúng `k-1` cách tô cột `i` khác màu, không tạo ra 3 cột trùng. ⇒ `(k-1)·total[i-1]` cách.
2. **Cột `i` cùng màu cột `i-1`.** Khi đó cột `i-1` **bắt buộc** khác cột `i-2`. Bỏ hai cột cuối, phần đầu là một cách hợp lệ `i-2` cột bất kỳ; cột `i-1` có `k-1` lựa chọn (khác cột `i-2`), cột `i` copy màu. ⇒ `(k-1)·total[i-2]` cách.

Hai nhóm rời nhau và phủ hết → `total[i] = (k-1)(total[i-1] + total[i-2])`. Base case `total[1] = k`, `total[2] = k²` đếm trực tiếp. Quy nạp theo `i` cho kết quả đúng mọi `n`.

**Invariant của vòng lặp `O(1)`:** đầu mỗi vòng, `prev2 = total[i-2]`, `prev1 = total[i-1]`.

### 10.1. Độ phức tạp và so sánh

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(k^n · n)` | `O(n)` | chỉ dùng kiểm tra |
| 2. Đệ quy + memo | `O(n)` | `O(n)` (cache + ngăn xếp) | gần với cách nghĩ toán |
| 3. DP bảng | `O(n)` | `O(n)` | dễ trace |
| 4. DP `O(1)` | `O(n)` | `O(1)` | **lời giải chính** |

Phép nhân số lớn: với `n <= 50` và đáp án `< 2^31`, mọi phép tính là số nhỏ, coi như `O(1)`.

## 11. Bộ test

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<30} PASS ({actual})")

sol = Solution()
check("mẫu 1: n=3, k=2", sol.numWays(3, 2), 6)
check("mẫu 2: n=1, k=1", sol.numWays(1, 1), 1)
check("mẫu 3: n=7, k=2", sol.numWays(7, 2), 42)
check("n=2: k^2", sol.numWays(2, 5), 25)
check("k=1, n=2: được 2 cột trùng", sol.numWays(2, 1), 1)
check("k=1, n=3: không thể", sol.numWays(3, 1), 0)
check("n=1, k lớn", sol.numWays(1, 10**5), 10**5)
check("n=5, k=3 (bảng trace)", sol.numWays(5, 3), 180)

mẫu 1: n=3, k=2                PASS (6)
mẫu 2: n=1, k=1                PASS (1)
mẫu 3: n=7, k=2                PASS (42)
n=2: k^2                       PASS (25)
k=1, n=2: được 2 cột trùng     PASS (1)
k=1, n=3: không thể            PASS (0)
n=1, k lớn                     PASS (100000)
n=5, k=3 (bảng trace)          PASS (180)


In [15]:
funcs = [num_ways_memo, num_ways_table, num_ways_o1, Solution().numWays]
for n in range(1, 8):
    for k in range(1, 5):
        expected = num_ways_brute(n, k)
        assert all(f(n, k) == expected for f in funcs), (n, k)
for n in range(1, 51):
    for k in [1, 2, 3, 10, 46]:
        assert len({f(n, k) for f in funcs}) == 1
print("Khớp brute force với n <= 7, k <= 4; các DP khớp nhau tới n = 50")

Khớp brute force với n <= 7, k <= 4; các DP khớp nhau tới n = 50


## 12. Lỗi người mới hay gặp

### Lỗi 1: Cấm cả **2** cột liên tiếp cùng màu

Đọc nhầm đề thành "không có 2 cột kề nhau cùng màu" → ra `k·(k-1)^(n-1)`.

In [16]:
def wrong_no_two(n, k):
    return k * (k - 1) ** (n - 1)

print("wrong_no_two(3, 2) =", wrong_no_two(3, 2), "| đúng =", num_ways_o1(3, 2))

wrong_no_two(3, 2) = 2 | đúng = 6


### Lỗi 2: `same[i] = same[i-1] + diff[i-1]` (quên ràng buộc 3 cột)

In [17]:
def wrong_same(n, k):
    same, diff = 0, k
    for _ in range(2, n + 1):
        same, diff = same + diff, (same + diff) * (k - 1)    # SAI: same[i] phải = diff[i-1]
    return same + diff

print("wrong_same(3, 2) =", wrong_same(3, 2), "| đúng =", num_ways_o1(3, 2), "(bản sai đếm luôn RRR, GGG)")

wrong_same(3, 2) = 8 | đúng = 6 (bản sai đếm luôn RRR, GGG)


### Lỗi 3: Sai base case `n = 2` (ví dụ dùng `k·(k-1)`)

Hai cột **được phép** cùng màu → `total[2] = k²`, không phải `k(k-1)`.

In [18]:
def wrong_base(n, k):
    if n == 1:
        return k
    p2, p1 = k, k * (k - 1)                                  # SAI
    for _ in range(3, n + 1):
        p2, p1 = p1, (k - 1) * (p1 + p2)
    return p1

print("wrong_base(2, 2) =", wrong_base(2, 2), "| đúng =", num_ways_o1(2, 2))

wrong_base(2, 2) = 2 | đúng = 4


### Lỗi 4: Cập nhật tuần tự làm mất giá trị cũ

```python
prev2 = prev1
prev1 = (k - 1) * (prev1 + prev2)   # prev2 lúc này đã bị ghi đè = prev1!
```

In [19]:
def wrong_update(n, k):
    if n == 1:
        return k
    prev2, prev1 = k, k * k
    for _ in range(3, n + 1):
        prev2 = prev1
        prev1 = (k - 1) * (prev1 + prev2)                    # SAI
    return prev1

print("wrong_update(4, 2) =", wrong_update(4, 2), "| đúng =", num_ways_o1(4, 2))

wrong_update(4, 2) = 16 | đúng = 10


**Sửa:** dùng gán đồng thời `prev2, prev1 = prev1, (k - 1) * (prev1 + prev2)` hoặc biến tạm.

### Lỗi 5: Quên `k = 1`

Với `k = 1`, `k - 1 = 0` nên mọi `total[i >= 3] = 0` — công thức tự xử lý đúng. Nhưng nếu ai đó "tối ưu" bằng cách chia cho `k - 1` thì sẽ lỗi chia cho 0. Đừng chia.

## 13. Mẹo nhận dạng pattern

- **Đếm số cách** + **ràng buộc cục bộ** (chỉ phụ thuộc vài phần tử cuối) → DP theo vị trí, trạng thái = "đuôi" của chuỗi.
- Công thức dạng `f(i) = a·f(i-1) + b·f(i-2)` → **giống Fibonacci**, tối ưu `O(1)` bộ nhớ; nếu `n` rất lớn → lũy thừa ma trận `O(log n)`.
- "Không quá `m` phần tử liên tiếp giống nhau" → trạng thái = độ dài chuỗi cùng màu ở cuối (`1..m`).

Bài liên quan: 70 Climbing Stairs, 198 House Robber, 256 Paint House, 265 Paint House II, 1220 Count Vowels Permutation, 935 Knight Dialer.

## 14. Biến thể

### 14.1. Không quá `m` cột liên tiếp cùng màu

Trạng thái `run[j]` = số cách mà đoạn cùng màu ở cuối dài đúng `j` (`1 <= j <= m`).
- Kéo dài đoạn: `run_new[j] = run[j-1]` (với `j >= 2`, copy màu).
- Đổi màu: `run_new[1] = (k-1) · sum(run)`.

Với `m = 2` ta có lại bài gốc.

In [20]:
def num_ways_max_run(n: int, k: int, m: int) -> int:
    run = [0] * (m + 1)       # run[1..m]
    run[1] = k
    for _ in range(2, n + 1):
        new = [0] * (m + 1)
        new[1] = (k - 1) * sum(run)
        for j in range(2, m + 1):
            new[j] = run[j - 1]
        run = new
    return sum(run)

def brute_max_run(n, k, m):
    def ok(c):
        r = 1
        for i in range(1, len(c)):
            r = r + 1 if c[i] == c[i - 1] else 1
            if r > m:
                return False
        return True
    return sum(1 for c in product(range(k), repeat=n) if ok(c))

print("m = 2 trùng bài gốc:", all(num_ways_max_run(n, k, 2) == num_ways_o1(n, k) for n in range(1, 20) for k in range(1, 6)))
print("m = 3, n = 6, k = 2:", num_ways_max_run(6, 2, 3), "| brute:", brute_max_run(6, 2, 3))

m = 2 trùng bài gốc: True
m = 3, n = 6, k = 2: 48 | brute: 48


### 14.2. Lũy thừa ma trận cho `n` khổng lồ (lấy mod)

`[t_i, t_{i-1}] = [[k-1, k-1], [1, 0]] · [t_{i-1}, t_{i-2}]`. Lũy thừa nhanh ma trận 2×2 → `O(log n)`.

In [21]:
def mat_mul(A, B, mod):
    return [[(A[0][0] * B[0][0] + A[0][1] * B[1][0]) % mod, (A[0][0] * B[0][1] + A[0][1] * B[1][1]) % mod],
            [(A[1][0] * B[0][0] + A[1][1] * B[1][0]) % mod, (A[1][0] * B[0][1] + A[1][1] * B[1][1]) % mod]]

def num_ways_matrix(n, k, mod=10**9 + 7):
    if n == 1:
        return k % mod
    M = [[k - 1, k - 1], [1, 0]]
    R = [[1, 0], [0, 1]]
    p = n - 2
    while p:
        if p & 1:
            R = mat_mul(R, M, mod)
        M = mat_mul(M, M, mod)
        p >>= 1
    return (R[0][0] * (k * k) + R[0][1] * k) % mod

assert all(num_ways_matrix(n, k) == num_ways_o1(n, k) % (10**9 + 7) for n in range(1, 60) for k in range(1, 6))
print("n = 10^18, k = 3 (mod 1e9+7):", num_ways_matrix(10**18, 3))

n = 10^18, k = 3 (mod 1e9+7): 823973457


## 15. Bài tập (tự làm — cell chỉ định nghĩa, không gọi)

**Bài 1 — Tính tay:** điền `total[1..6]` với `k = 3`.

**Bài 2 — Điền chỗ trống:** DP `same/diff`.

**Bài 3 — Tự viết bản `O(1)`** bằng công thức gộp.

**Bài 4 — Biến thể:** cột đầu tiên **bắt buộc** màu 0 (chỉ các cột sau được tự do). Đếm số cách.

**Bài 5 — Không có 2 cột kề nhau cùng màu** và **cột đầu khác cột cuối** (hàng rào vòng tròn, `n >= 2`).

**Bài 6 — Mini challenge:** giống bài gốc nhưng **không quá `m`** cột liên tiếp cùng màu, viết bằng **tổng trượt** để mỗi bước `O(1)` thay vì `O(m)`.

In [22]:
# Bài 1 — TODO
predict_k3 = None   # list [total1, total2, ..., total6]
# assert predict_k3 == [Solution().numWays(i, 3) for i in range(1, 7)]

In [23]:
def exercise_same_diff(n: int, k: int) -> int:
    same, diff = 0, k
    for _ in range(2, n + 1):
        # TODO: cập nhật đồng thời same, diff
        pass
    return same + diff

# assert exercise_same_diff(7, 2) == 42

In [24]:
def exercise_o1(n: int, k: int) -> int:
    # TODO
    pass

# assert [exercise_o1(n, 2) for n in range(1, 8)] == [2, 4, 6, 10, 16, 26, 42]

In [25]:
def exercise_first_fixed(n: int, k: int) -> int:
    # TODO: gợi ý đối xứng màu
    pass

# assert exercise_first_fixed(3, 2) == 3

In [26]:
def exercise_circular_no_two(n: int, k: int) -> int:
    # TODO: công thức (k-1)^n + (-1)^n (k-1), hoặc DP theo "cột cuối bằng/khác cột đầu"
    pass

# assert exercise_circular_no_two(3, 3) == 6 and exercise_circular_no_two(4, 2) == 2

In [27]:
def exercise_max_run_fast(n: int, k: int, m: int) -> int:
    # TODO: total[i] = (k-1) * (total[i-1] + ... + total[i-m]) với base hợp lý
    pass

# assert all(exercise_max_run_fast(n, k, m) == num_ways_max_run(n, k, m) for n in range(1, 15) for k in range(1, 4) for m in range(1, 5))

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `total[1] = 3`, `total[2] = 9` | nhân `k - 1 = 2` | `total[3] = 2·(9+3)` |
| 2 | xem mục 4 | `same` mới = `diff` cũ | `same, diff = diff, (same + diff) * (k - 1)` |
| 3 | hai biến trượt | gán đồng thời | base `k`, `k*k` |
| 4 | mỗi màu cột đầu cho số cách như nhau | đáp án gốc chia `k` | `numWays(n, k) // k` |
| 5 | đếm chuỗi thẳng rồi trừ chuỗi có đầu = cuối | hoặc DP 2 trạng thái | `(k-1)^n + (-1)^n·(k-1)` |
| 6 | chia theo độ dài đoạn cuối `j = 1..m` | `total[i] = Σ_{j=1..m} w(i-j)` với `w(0) = k`, `w(t) = (k-1)·total[t]` | dùng tổng trượt: cộng phần tử mới, trừ phần tử rơi khỏi cửa sổ |

## 17. Đáp án tham khảo

In [28]:
answer_k3 = [3, 9, 24, 66, 180, 492]

def answer_same_diff(n, k):
    same, diff = 0, k
    for _ in range(2, n + 1):
        same, diff = diff, (same + diff) * (k - 1)
    return same + diff

def answer_o1(n, k):
    if n == 1:
        return k
    p2, p1 = k, k * k
    for _ in range(3, n + 1):
        p2, p1 = p1, (k - 1) * (p1 + p2)
    return p1

def answer_first_fixed(n, k):
    return answer_o1(n, k) // k          # đổi tên màu: mỗi màu cột đầu có số cách bằng nhau

def answer_circular_no_two(n, k):
    return (k - 1) ** n + (-1) ** n * (k - 1)

def answer_max_run_fast(n, k, m):
    # Chia theo độ dài j (1..m) của đoạn cùng màu ở cuối: đoạn đó bắt đầu sau t = i - j cột.
    # w(t) = số cách tô t cột đầu + chọn màu cho đoạn mới: w(0) = k, w(t > 0) = (k - 1) * total[t]
    # total[i] = w(i-1) + w(i-2) + ... + w(i-m)   (bỏ các chỉ số âm) -> tổng trượt
    total = [0] * (n + 1)
    window = 0                            # tổng w(i-1) + ... + w(i-m)
    w = [0] * (n + 1)
    w[0] = k
    for i in range(1, n + 1):
        window += w[i - 1]
        if i - 1 - m >= 0:
            window -= w[i - 1 - m]
        total[i] = window
        w[i] = (k - 1) * total[i]
    return total[n]

def brute_circular_no_two(n, k):
    return sum(1 for c in product(range(k), repeat=n)
               if all(c[i] != c[(i + 1) % n] for i in range(n)))

assert answer_k3 == [Solution().numWays(i, 3) for i in range(1, 7)]
assert answer_same_diff(7, 2) == 42
assert [answer_o1(n, 2) for n in range(1, 8)] == [2, 4, 6, 10, 16, 26, 42]
assert answer_first_fixed(3, 2) == 3
assert all(answer_first_fixed(n, k) == sum(1 for c in product(range(k), repeat=n) if c[0] == 0 and not has_three_in_row(c))
           for n in range(1, 7) for k in range(1, 4))
assert answer_circular_no_two(3, 3) == 6 and answer_circular_no_two(4, 2) == 2
assert all(answer_circular_no_two(n, k) == brute_circular_no_two(n, k) for n in range(2, 7) for k in range(1, 4))
assert all(answer_max_run_fast(n, k, m) == num_ways_max_run(n, k, m) for n in range(1, 15) for k in range(1, 4) for m in range(1, 5))
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def numWays(self, n: int, k: int) -> int:
        if n == 1:
            return k
        prev2, prev1 = k, k * k
        for _ in range(3, n + 1):
            prev2, prev1 = prev1, (k - 1) * (prev1 + prev2)
        return prev1
```

**Câu thần chú:** *"Cột cuối khác cột trước: (k-1)·total[i-1]; cột cuối cùng cột trước: (k-1)·total[i-2]."*

**Checklist:**
- [ ] Đọc đúng: cấm **3** cột liên tiếp, 2 cột thì được.
- [ ] Base: `total[1] = k`, `total[2] = k*k`.
- [ ] Gán đồng thời khi trượt hai biến.
- [ ] `k = 1` cho `0` khi `n >= 3` (công thức tự đúng).
- [ ] Tên method `numWays`, trả `int`.